This notebook is for analysing the NHS monthly admission data to understand the distribution

In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
# Get list of all CSV files in data dir
files = [file for file in os.listdir('../data') if file.endswith('.csv')]

# Get list of dataframes from all CSV files
dfs = [pd.read_csv(f'../data/{file}') for file in files]

for i, df in enumerate(dfs):
    print(f"Dataframe {i} shape: {df.shape}") # All dataframes have 22 columns and similar rows


# Merge all dataframes into one
df = pd.concat(dfs, ignore_index=True)

Dataframe 0 shape: (199, 22)
Dataframe 1 shape: (202, 22)
Dataframe 2 shape: (202, 22)
Dataframe 3 shape: (201, 22)
Dataframe 4 shape: (200, 22)
Dataframe 5 shape: (198, 22)
Dataframe 6 shape: (202, 22)
Dataframe 7 shape: (198, 22)
Dataframe 8 shape: (202, 22)
Dataframe 9 shape: (198, 22)
Dataframe 10 shape: (197, 22)
Dataframe 11 shape: (198, 22)


In [3]:
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 2397 entries, 0 to 2396
Data columns (total 22 columns):
 #   Column                                                      Non-Null Count  Dtype
---  ------                                                      --------------  -----
 0   Period                                                      2397 non-null   str  
 1   Org Code                                                    2397 non-null   str  
 2   Parent Org                                                  2397 non-null   str  
 3   Org name                                                    2397 non-null   str  
 4   A&E attendances Type 1                                      2397 non-null   int64
 5   A&E attendances Type 2                                      2397 non-null   int64
 6   A&E attendances Other A&E Department                        2397 non-null   int64
 7   A&E attendances Booked Appointments Type 1                  2397 non-null   int64
 8   A&E attendances Booked Appoin

None

## Cleaning

### Date Parsing

In [4]:
# Looking at the df information, the period column needs to be parsed as a date. 

# Checking the unique values in the period column 
display(df['Period'].unique())

# Remove the total rows, where the period has total.
df = df[~df['Period'].str.contains('total', case=False, na=False)]


<StringArray>
[    'MSitAE-APRIL-2025',                 'TOTAL',      'MSitAE-JULY-2025',
 'MSitAE-SEPTEMBER-2025',       'MSitAE-MAY-2025',     'MSitAE-MARCH-2026',
  'MSitAE-DECEMBER-2025',                'TOTAL ',    'MSitAE-AUGUST-2025',
  'MSitAE-NOVEMBER-2025',      'MSitAE-JUNE-2025',  'MSitAE-FEBRUARY-2026',
   'MSitAE-OCTOBER-2025',   'MSitAE-JANUARY-2026',                 'Total']
Length: 15, dtype: str

In [5]:

df['Period'] = df['Period'].str.replace('MSitAE-', '') # Remove the MSitAE- prefix from the period column
df['Period'] = pd.to_datetime(df['Period'], format='%B-%Y') # Parse the values, like 'APRIL-2025', as dates

### Whitespace removal

In [6]:
# Removing the whitespace from all rows
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

### Column name cleaning

In [7]:
display(df.columns)

# remove the special chars from colum names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace("&", "_and_", regex=False)
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

# for all column names, replace more than one underscore with a single underscore
df.columns = (
    df.columns
      .str.replace(r'_+', '_', regex=True)
)
display(df.columns)

Index(['Period', 'Org Code', 'Parent Org', 'Org name',
       'A&E attendances Type 1', 'A&E attendances Type 2',
       'A&E attendances Other A&E Department',
       'A&E attendances Booked Appointments Type 1',
       'A&E attendances Booked Appointments Type 2',
       'A&E attendances Booked Appointments Other Department',
       'Attendances over 4hrs Type 1', 'Attendances over 4hrs Type 2',
       'Attendances over 4hrs Other Department',
       'Attendances over 4hrs Booked Appointments Type 1',
       'Attendances over 4hrs Booked Appointments Type 2',
       'Attendances over 4hrs Booked Appointments Other Department',
       'Patients who have waited 4-12 hs from DTA to admission',
       'Patients who have waited 12+ hrs from DTA to admission',
       'Emergency admissions via A&E - Type 1',
       'Emergency admissions via A&E - Type 2',
       'Emergency admissions via A&E - Other A&E department',
       'Other emergency admissions'],
      dtype='str')

Index(['period', 'org_code', 'parent_org', 'org_name',
       'a_and_e_attendances_type_1', 'a_and_e_attendances_type_2',
       'a_and_e_attendances_other_a_and_e_department',
       'a_and_e_attendances_booked_appointments_type_1',
       'a_and_e_attendances_booked_appointments_type_2',
       'a_and_e_attendances_booked_appointments_other_department',
       'attendances_over_4hrs_type_1', 'attendances_over_4hrs_type_2',
       'attendances_over_4hrs_other_department',
       'attendances_over_4hrs_booked_appointments_type_1',
       'attendances_over_4hrs_booked_appointments_type_2',
       'attendances_over_4hrs_booked_appointments_other_department',
       'patients_who_have_waited_4_12_hs_from_dta_to_admission',
       'patients_who_have_waited_12+_hrs_from_dta_to_admission',
       'emergency_admissions_via_a_and_e_type_1',
       'emergency_admissions_via_a_and_e_type_2',
       'emergency_admissions_via_a_and_e_other_a_and_e_department',
       'other_emergency_admissions'],